In [1]:
import gmsh

import numpy as np
from mpi4py import MPI


# Visualization

import pandas as pd

In [2]:
file ='/Users/straniero/Documents/Dphil/muon_collider/elements.csv'
df = pd.read_csv(file, index_col=0)
bore_radius = df.loc['Bore radius', 'pos']/100
Length = 37.47405725/100
Iris_a = df.loc['Iris A', 'pos']/100
Iris_b = df.loc['Iris B', 'pos']/100
Iris_center_y = df.loc['Iris center y', 'pos']/100
Iris_center_x = 0.0
iris_pos1_y = df.loc['iris_p1_y', 'pos']/100
iris_pos1_x = df.loc['iris_p1_x', 'pos']/100
iris_pos2_y = df.loc['iris_p2_y', 'pos']/100
iris_pos2_x = df.loc['iris_p2_x', 'pos']/100


Dome_A = df.loc['Dome A', 'pos']/100
Dome_B = df.loc['Dome B', 'pos']/100
Dome_center_x = Length/2
Dome_center_y = df.loc['Dome center y', 'pos']/100
dome_pos0_x = 12.62750856/100
dome_pos1_y = df.loc['dome_p1_y', 'pos']/100
dome_pos1_x = df.loc['dome_p1_x', 'pos']/100
dome_pos2_y = df.loc['dome_p2_y', 'pos']/100
dome_pos2_x = df.loc['dome_p2_x', 'pos']/100



iris_p1 = [iris_pos1_x + Iris_center_x, iris_pos1_y + Iris_center_y]
iris_p2 = [iris_pos2_x + Iris_center_x, iris_pos2_y + Iris_center_y]

dome_p0 = [Dome_center_x-dome_pos2_x, dome_pos2_y+Dome_center_y]
dome_p1 = [Dome_center_x,             Dome_center_y+dome_pos1_y]
dome_p2 = [Dome_center_x+dome_pos2_x, dome_pos2_y+Dome_center_y]


In [3]:
Dome_center_x

0.18737028625000002

In [4]:
gmsh.initialize()
gmsh.option.setNumber("General.Terminal", 1)
gmsh.model.add("ell_cav")

p1 = gmsh.model.geo.addPoint(0, 0, 0)
p2 = gmsh.model.geo.addPoint(0, bore_radius, 0)
p3 = gmsh.model.geo.addPoint(iris_p2[0], iris_p2[1], 0)
p4 = gmsh.model.geo.addPoint(dome_p0[0], dome_p0[1], 0)
p5 = gmsh.model.geo.addPoint(dome_p1[0], dome_p1[1], 0)
p6 = gmsh.model.geo.addPoint(dome_p2[0], dome_p2[1], 0)
p7 = gmsh.model.geo.addPoint(Length-iris_pos2_x, iris_p2[1], 0)
p8 = gmsh.model.geo.addPoint(Length, bore_radius, 0)
p9 = gmsh.model.geo.addPoint(Length, 0, 0)

iris_center = gmsh.model.geo.addPoint(Iris_center_x, Iris_center_y, 0)
iris_maj = gmsh.model.geo.addPoint(Iris_center_x , Iris_center_y-1, 0)
iris_center2 = gmsh.model.geo.addPoint(Length , Iris_center_y, 0)
iris_maj2 = gmsh.model.geo.addPoint(Length , Iris_center_y-1/100, 0)
dome_center = gmsh.model.geo.addPoint(Dome_center_x, Dome_center_y, 0)
dome_maj = gmsh.model.geo.addPoint(Dome_center_x+1/100, Dome_center_y, 0)
#lines

l1 = gmsh.model.geo.addLine(p1, p2)
l2 = gmsh.model.geo.addEllipseArc(p2, iris_center, iris_maj, p3)
l3 = gmsh.model.geo.addLine(p3, p4)
l4 = gmsh.model.geo.addEllipseArc(p4, dome_center, dome_maj, p5)
l5 = gmsh.model.geo.addEllipseArc(p5, dome_center, dome_maj, p6)
l6 = gmsh.model.geo.addLine(p6, p7)
l7 = gmsh.model.geo.addEllipseArc(p7, iris_center2, iris_maj2, p8)
l8 = gmsh.model.geo.addLine(p8, p9)
l9 = gmsh.model.geo.addLine(p9, p1)

cl1 = gmsh.model.geo.addCurveLoop([l1, l2, l3, l4, l5, l6, l7, l8, l9])
s1 = gmsh.model.geo.addPlaneSurface([cl1])

# Synchronize and create physical groups so dolfinx.gmshio can detect cell types
gmsh.model.geo.synchronize()
surf_tag = gmsh.model.addPhysicalGroup(2, [s1])
gmsh.model.setPhysicalName(2, surf_tag, "domain")
line_tags = [l1, l2, l3, l4, l5, l6, l7, l8]
bnd_tag = gmsh.model.addPhysicalGroup(1, line_tags)
gmsh.model.setPhysicalName(1, bnd_tag, "boundary")

# Set mesh precision (adjust mesh_size for finer/coarser mesh)
mesh_size = 0.005  # decrease for finer mesh
gmsh.model.mesh.setSize(gmsh.model.getEntities(0), mesh_size)

gmsh.model.mesh.generate(2)

msh_filename = "ell_shape_small.msh"
gmsh.write(msh_filename)
print(f"Saved mesh to {msh_filename}")




# finalize gmsh
gmsh.finalize()

Info    : Meshing 1D...
Info    : [  0%] Meshing curve 1 (Line)
Info    : [ 20%] Meshing curve 2 (Ellipse)
Info    : [ 30%] Meshing curve 3 (Line)
Info    : [ 40%] Meshing curve 4 (Ellipse)
Info    : [ 50%] Meshing curve 5 (Ellipse)
Info    : [ 60%] Meshing curve 6 (Line)
Info    : [ 70%] Meshing curve 7 (Ellipse)
Info    : [ 80%] Meshing curve 8 (Line)
Info    : [ 90%] Meshing curve 9 (Line)
Info    : Done meshing 1D (Wall 0.00219275s, CPU 0.003642s)
Info    : Meshing 2D...
Info    : Meshing surface 1 (Plane, Frontal-Delaunay)
Info    : Done meshing 2D (Wall 0.0439431s, CPU 0.069951s)
Info    : 3816 nodes 7633 elements
Info    : Writing 'ell_shape_small.msh'...
Saved mesh to ell_shape_small.msh
Info    : Done writing 'ell_shape_small.msh'


In [3]:
np.cos(np.pi)

np.float64(-1.0)